In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movie_data = pd.read_csv("../../datasets/processed/movie_data.csv")

movie_data.head()

,userId,movieId,rating,timestamp,title,genres,year,clean_title,average_rating,rating_count,content
0,1,1,4.0,964982703,Toy Story (1995),adventure animation children comedy fantasy,1995,Toy Story,3.920930,215,Toy Story adventure animation children comedy ...
1,1,3,4.0,964981247,Grumpier Old Men (1995),comedy romance,1995,Grumpier Old Men,3.259615,52,Grumpier Old Men comedy romance
2,1,6,4.0,964982224,Heat (1995),action crime thriller,1995,Heat,3.946078,102,Heat action crime thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),mystery thriller,1995,Seven (a.k.a. Se7en),3.975369,203,Seven (a.k.a. Se7en) mystery thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",crime mystery thriller,1995,"Usual Suspects, The",4.237745,204,"Usual Suspects, The crime mystery thriller"


In [3]:
print(movie_data.shape)

movie_data.info()

(90274, 11)
<class 'pandas.DataFrame'>
RangeIndex: 90274 entries, 0 to 90273
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   userId          90274 non-null  int64  
 1   movieId         90274 non-null  int64  
 2   rating          90274 non-null  float64
 3   timestamp       90274 non-null  int64  
 4   title           90274 non-null  str    
 5   genres          90274 non-null  str    
 6   year            90274 non-null  int64  
 7   clean_title     90274 non-null  str    
 8   average_rating  90274 non-null  float64
 9   rating_count    90274 non-null  int64  
 10  content         90274 non-null  str    
dtypes: float64(2), int64(5), str(4)
memory usage: 16.3 MB


In [4]:
# Reset the index of the dataframe to be continuous
movie_data = movie_data.reset_index(drop=True)

In [5]:
movie_data["content"] = movie_data["content"].fillna("")

In [6]:
movie_data = movie_data.drop_duplicates(
    subset="movieId"
)

movie_data.shape

(3650, 11)

In [7]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

In [8]:
tfidf_matrix = tfidf.fit_transform(
    movie_data["content"]
)

print(tfidf_matrix.shape)

(3650, 4061)


In [9]:
print(len(tfidf.vocabulary_))

4061


In [10]:
cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print(cosine_sim.shape)

(3650, 3650)


In [11]:
indices = pd.Series(
    movie_data.index,
    index=movie_data["clean_title"]
).drop_duplicates()

In [12]:
def recommend_movies(title, top_n=10):

    if title not in indices:
        return "Movie not found."

    idx = indices[title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    movie_indices = [
        i[0]
        for i in similarity_scores
    ]

    return movie_data[
        [
            "clean_title",
            "genres",
            "average_rating",
            "rating_count"
        ]
    ].iloc[movie_indices]

In [13]:
recommend_movies("Toy Story")

,clean_title,genres,average_rating,rating_count
857,Toy Story 2,adventure animation children comedy fantasy,3.860825,97
1560,Toy Story 3,adventure animation children comedy fantasy imax,4.109091,55
7824,"Story of Us, The",comedy drama,2.400000,5
137,"NeverEnding Story, The",adventure children fantasy,3.581395,43
2487,L.A. Story,comedy romance,3.479167,24
1114,Up,adventure animation children drama,4.004762,105
2611,"Christmas Story, A",children comedy,3.972727,55
289,"NeverEnding Story III, The",adventure children fantasy,2.000000,7
3205,Shrek the Third,adventure animation children comedy fantasy,3.023810,21
1457,Inside Out,adventure animation children comedy drama fantasy,3.813953,43


In [14]:
movie_data = movie_data.reset_index(drop=True)

indices = pd.Series(movie_data.index, index=movie_data['clean_title']).drop_duplicates()

print("Index and indices mapping updated successfully!")

Index and indices mapping updated successfully!


In [15]:
movie_data.groupby(
    "genres"
)["average_rating"].mean().sort_values(
    ascending=False
).head(10)

genres
drama mystery war                          4.400000
animation comedy romance                   4.375000
documentary war                            4.307692
action animation drama fantasy sci-fi      4.300000
animation comedy fantasy sci-fi            4.277778
documentary imax                           4.250000
film-noir romance thriller                 4.250000
action adventure drama thriller western    4.250000
crime thriller war                         4.250000
action crime drama imax                    4.238255
Name: average_rating, dtype: float64

In [16]:
movie_data["genres"].value_counts().head(20)

genres
comedy                       331
drama                        275
comedy drama                 144
comedy romance               143
drama romance                122
comedy drama romance         115
drama thriller                73
crime drama                   54
crime drama thriller          49
comedy crime                  47
action adventure sci-fi       45
documentary                   39
action crime thriller         38
drama war                     37
children comedy               37
horror thriller               37
action adventure thriller     35
horror                        34
action comedy                 34
action sci-fi thriller        33
Name: count, dtype: int64

In [17]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:50])

['000' '10' '1000' '101' '102' '11' '1138' '12' '127' '13' '13th' '1408'
 '15' '16' '17' '19' '1984' '20' '200' '2000' '2001' '2010' '2012' '2046'
 '2049' '21' '22' '23' '24' '25th' '27' '28' '2nd' '30' '300' '3000' '33'
 '34th' '37th' '39' '3d' '40' '400' '42nd' '451' '47' '48' '49' '50' '500']


In [18]:
import joblib

joblib.dump(
    tfidf,
    "../../ai-service/models/tfidf_vectorizer.pkl"
)

joblib.dump(
    cosine_sim,
    "../../ai-service/models/cosine_similarity.pkl"
)

print("Models Saved Successfully")

Models Saved Successfully


In [19]:
print("="*60)

print("TF-IDF FEATURE ENGINEERING COMPLETED")

print("="*60)

print("Movies :", len(movie_data))

print("Vocabulary :", len(tfidf.vocabulary_))

print("Similarity Matrix :", cosine_sim.shape)

TF-IDF FEATURE ENGINEERING COMPLETED
Movies : 3650
Vocabulary : 4061
Similarity Matrix : (3650, 3650)
